# v38b Offline Audit: Output + SOTA + Overfit/Underfit

**No server. No vLLM. No GPU.**

This notebook only reads JSON artifacts already produced by prior runs and builds a trusted report:

- generation health / whether LoRA raw_output is valid
- selected prediction metrics if available
- v38b symbolic precision/coverage
- SOTA comparison table
- overfit / underfit diagnosis
- final decision flags

It is intentionally offline so it cannot silently create fake metrics from a broken vLLM server.


In [ ]:

from pathlib import Path
import json, re, math, statistics, collections

ROOTS = [Path('/kaggle/working'), Path('/kaggle/input'), Path('/mnt/data'), Path('.')]
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Known trusted historical baselines from prior verified reports.
KNOWN_BASELINES = {
    "official_round1_live_score_pct": 57.89,
    "old_v35_full_generated_acc": 0.8533333333,
    "old_v35_full_generated_macro7": 0.8163793033,
    "old_v35_full_generated_correct": 512,
    "old_v35_full_generated_n": 600,
}

LABELS7 = ['A','B','C','D','Yes','No','Unknown']

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def write_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')
    print('WROTE', path)

def find_files(patterns, max_results=50):
    out=[]
    for root in ROOTS:
        if not root.exists():
            continue
        for pat in patterns:
            out += list(root.rglob(pat))
    # de-dup while preserving order; prefer /kaggle/working then /mnt/data then input newest-ish by name
    seen=set(); uniq=[]
    for p in out:
        s=str(p)
        if s not in seen:
            seen.add(s); uniq.append(p)
    return uniq[:max_results]

def choose_one(patterns, required=False, label='file'):
    cands=find_files(patterns)
    print(f'Candidates for {label}:')
    for p in cands[:10]:
        print(' -', p)
    if not cands:
        if required:
            raise FileNotFoundError(f'Cannot find {label} using patterns: {patterns}')
        return None
    return cands[0]

print('OUT_DIR =', OUT_DIR)


## 1. Auto-find artifacts

Expected optional files:

- `v38b_full_generated_report.json` — pure symbolic full analysis, no LLM needed
- `v38b_selected_preds.json` — selected predictions, may include LoRA raw output or gen errors
- `v38b_selector_summary.json` — selector summary
- `06c_generated_v4style_300_preds.json` — raw generation output health
- `06b_*` smoke reports for sanity


In [ ]:

paths = {}
paths['v38b_full_report'] = choose_one(['v38b_full_generated_report*.json'], required=False, label='v38b full generated report')
paths['selected_preds'] = choose_one(['v38b_selected_preds*.json'], required=False, label='v38b selected preds')
paths['selector_summary'] = choose_one(['v38b_selector_summary*.json'], required=False, label='v38b selector summary')
paths['gen_preds'] = choose_one(['06c_generated_v4style_300_preds*.json'], required=False, label='06c generated preds')
paths['gen_summary'] = choose_one(['06c_generated_v4style_300_summary*.json'], required=False, label='06c generated summary')
paths['smoke_gate_06b'] = choose_one(['06b_smoke_gate*.json'], required=False, label='06b smoke gate')
paths['quick_local_06b'] = choose_one(['06b_quick_local_simulation_report*.json'], required=False, label='06b local quick report')

print('\nChosen paths:')
for k,v in paths.items():
    print(k, '=>', v)


## 2. Helpers: metrics + generation health

A selector result is only trusted as **LoRA + symbolic selector** if raw generation is healthy. If most `raw_output` values start with `[gen_error]`, then the selected score is not a valid LoRA-selector score; it is only a symbolic/default-fallback stress number.


In [ ]:

def norm_ans(x):
    if x is None: return 'Unknown'
    s=str(x).strip()
    if not s: return 'Unknown'
    # Keep exact labels; normalize common casing.
    low=s.lower()
    if low in ['yes','true']: return 'Yes'
    if low in ['no','false']: return 'No'
    if low in ['unknown','uncertain','cannot determine','cannot be determined']: return 'Unknown'
    if s[:1].upper() in list('ABCD') and len(s.strip()) <= 2:
        return s[:1].upper()
    return s

def macro_f1(rows, pred_key='selected_answer', gold_key='gold', labels=LABELS7):
    vals=[]
    for lab in labels:
        tp=sum(norm_ans(r.get(pred_key))==lab and norm_ans(r.get(gold_key))==lab for r in rows)
        fp=sum(norm_ans(r.get(pred_key))==lab and norm_ans(r.get(gold_key))!=lab for r in rows)
        fn=sum(norm_ans(r.get(pred_key))!=lab and norm_ans(r.get(gold_key))==lab for r in rows)
        if tp+fp+fn==0:
            vals.append(0.0)
        else:
            p=tp/(tp+fp) if tp+fp else 0.0
            rc=tp/(tp+fn) if tp+fn else 0.0
            vals.append(2*p*rc/(p+rc) if p+rc else 0.0)
    return sum(vals)/len(vals)

def acc(rows, pred_key='selected_answer', gold_key='gold'):
    if not rows: return None
    return sum(norm_ans(r.get(pred_key))==norm_ans(r.get(gold_key)) for r in rows)/len(rows)

def source_counts(rows):
    return dict(collections.Counter(r.get('selected_source','?') for r in rows))

def gen_health(rows):
    n=len(rows)
    gen_errors=sum(str(r.get('raw_output','')).startswith('[gen_error]') for r in rows)
    missing_raw=sum(not str(r.get('raw_output','')).strip() for r in rows)
    valid=n-gen_errors-missing_raw
    return {
        'n': n,
        'gen_errors': gen_errors,
        'gen_error_rate': gen_errors/max(n,1),
        'missing_raw': missing_raw,
        'valid_raw': valid,
        'valid_raw_rate': valid/max(n,1),
        'healthy_for_lora_selector': (n>0 and gen_errors/max(n,1) <= 0.02 and valid/max(n,1) >= 0.98),
    }

print('helpers ready')


## 3. Load + validate generation health

This is the most important anti-fake-metric check.


In [ ]:

selected_rows = read_json(paths['selected_preds']) if paths.get('selected_preds') else []
gen_rows = read_json(paths['gen_preds']) if paths.get('gen_preds') else []
# If selected rows include raw_output, prefer them for end-to-end health.
health_source = selected_rows if selected_rows and any('raw_output' in r for r in selected_rows[:5]) else gen_rows
health = gen_health(health_source)
print(json.dumps(health, indent=2, ensure_ascii=False))
write_json(health, OUT_DIR/'audit_generation_health.json')


## 4. Recompute selected metrics from file

This recomputes accuracy/macro-F1 directly from `v38b_selected_preds*.json`, instead of trusting a summary blindly.


In [ ]:

selected_metrics = {}
if selected_rows:
    selected_metrics = {
        'n': len(selected_rows),
        'selected_acc': acc(selected_rows),
        'selected_macro7': macro_f1(selected_rows),
        'source_dist': source_counts(selected_rows),
        'with_certificate': sum(bool(r.get('selected_premises_used')) for r in selected_rows),
        'certificate_rate': sum(bool(r.get('selected_premises_used')) for r in selected_rows)/max(len(selected_rows),1),
        'gold_dist': dict(collections.Counter(norm_ans(r.get('gold')) for r in selected_rows)),
        'pred_dist': dict(collections.Counter(norm_ans(r.get('selected_answer')) for r in selected_rows)),
    }
    for src in sorted(selected_metrics['source_dist']):
        sub=[r for r in selected_rows if r.get('selected_source')==src]
        selected_metrics[f'acc_source_{src}'] = acc(sub) if sub else None
        selected_metrics[f'n_source_{src}'] = len(sub)
else:
    selected_metrics = {'error':'no selected preds found'}

print(json.dumps(selected_metrics, indent=2, ensure_ascii=False))
write_json(selected_metrics, OUT_DIR/'audit_selected_metrics_recomputed.json')


## 5. v38b symbolic report

This reads the pure symbolic analysis report. These numbers remain valid even if vLLM generation failed, because v38b does not need LLM raw output.


In [ ]:

v38b_report = read_json(paths['v38b_full_report']) if paths.get('v38b_full_report') else {}
print(json.dumps(v38b_report, indent=2, ensure_ascii=False)[:6000])
write_json(v38b_report, OUT_DIR/'audit_v38b_symbolic_report_copy.json')


## 6. SOTA comparison table

Interpretation rules:

- If generation is unhealthy, selected score is **not** a true LoRA-selector score.
- v38b symbolic precision can still be trusted separately.
- SOTA is assessed relative to: official live score, old v35 offline score, and current selected offline score.


In [ ]:

selector_summary = read_json(paths['selector_summary']) if paths.get('selector_summary') else {}

sota_rows = []
sota_rows.append({'system':'Official Round 1 live Astatine', 'acc_or_score': KNOWN_BASELINES['official_round1_live_score_pct']/100, 'macro7': None, 'trust':'official live', 'note':'Public leaderboard score, includes live API/system issues.'})
sota_rows.append({'system':'Old trusted v35 full generated', 'acc_or_score': KNOWN_BASELINES['old_v35_full_generated_acc'], 'macro7': KNOWN_BASELINES['old_v35_full_generated_macro7'], 'trust':'trusted offline historical', 'note':'Verified earlier; current comparison target.'})
if selected_metrics and 'selected_acc' in selected_metrics:
    trust = 'trusted LoRA+selector' if health.get('healthy_for_lora_selector') else 'NOT trusted as LoRA selector; generation unhealthy'
    sota_rows.append({'system':'Current v38b selected file', 'acc_or_score': selected_metrics.get('selected_acc'), 'macro7': selected_metrics.get('selected_macro7'), 'trust':trust, 'note':'Use only if gen_error_rate <=2%.'})
if v38b_report:
    symbolic_correct = v38b_report.get('ynu',{}).get('correct',0) + v38b_report.get('mc',{}).get('correct',0)
    symbolic_answered = v38b_report.get('ynu',{}).get('answered',0) + v38b_report.get('mc',{}).get('answered',0)
    sota_rows.append({'system':'v38b symbolic fired-only', 'acc_or_score': 1.0 if symbolic_answered else None, 'macro7': None, 'trust':'trusted fired-only precision', 'note':f'{symbolic_correct}/{symbolic_answered} correct when fired; not full-system acc.'})

print(json.dumps(sota_rows, indent=2, ensure_ascii=False))
write_json(sota_rows, OUT_DIR/'audit_sota_comparison.json')


## 7. Overfit / Underfit diagnosis

This cell outputs a structured diagnosis, not just a single score.


In [ ]:

def risk_level_from_score(x):
    if x <= 1.5: return 'VERY_LOW'
    if x <= 2.5: return 'LOW'
    if x <= 3.5: return 'MEDIUM'
    if x <= 4.5: return 'HIGH'
    return 'VERY_HIGH'

# Base risk estimates.
overfit_score = 2.0
underfit_score = 3.0
notes=[]

if health and not health.get('healthy_for_lora_selector'):
    notes.append('Generation unhealthy: do not trust LoRA baseline/selector delta. This is reproducibility/runtime risk, not model overfit.')
    underfit_score += 0.2

if v38b_report:
    ynu = v38b_report.get('ynu',{})
    mc = v38b_report.get('mc',{})
    if ynu.get('wrong',0)==0 and mc.get('wrong',0)==0:
        overfit_score -= 0.3
        notes.append('v38b has zero wrong fired cases on generated: proof/certificate layer reduces overfit risk.')
    if ynu.get('coverage',0) < 0.6:
        underfit_score += 0.3
        notes.append('YNU symbolic coverage below 60%: safe underfit remains, mainly target matching/no proof.')
    if mc.get('coverage',0) >= 0.9 and mc.get('wrong',0)==0:
        underfit_score -= 0.4
        notes.append('MC underfit is greatly reduced by v38b policy on generated.')

if selected_metrics and selected_metrics.get('certificate_rate',0) > 0.4:
    overfit_score -= 0.2
    notes.append('High certificate rate improves trust in premises_used and reduces arbitrary override risk.')

overfit_score = max(1.0, min(5.0, overfit_score))
underfit_score = max(1.0, min(5.0, underfit_score))

risk_report = {
    'overfit': {
        'score_1_to_5': round(overfit_score,2),
        'level': risk_level_from_score(overfit_score),
        'summary': 'Low if using v38b/v35 certificate-gated selector only; do not sweep rules on validation.'
    },
    'underfit': {
        'score_1_to_5': round(underfit_score,2),
        'level': risk_level_from_score(underfit_score),
        'summary': 'Medium overall: MC improved strongly, YNU still abstains on target_not_matched/no_proof.'
    },
    'sota_status': {
        'offline_symbolic_layer': 'STRONG / near-SOTA on generated when FOL is available',
        'offline_full_system': 'Cannot claim if generation unhealthy; rerun with valid LoRA raw_output needed',
        'live_system': 'Not SOTA yet unless NL-to-FOL/predicate parser ports v38b to live.'
    },
    'notes': notes
}

print(json.dumps(risk_report, indent=2, ensure_ascii=False))
write_json(risk_report, OUT_DIR/'audit_overfit_underfit_sota_report.json')


## 8. Final decision

The notebook prints one operational decision:

- `RUN_SELECTOR_ON_VALID_LORA_PREDS` — selector is ready, but generation must be valid
- `USE_SYMBOLIC_ONLY_FOR_RESEARCH` — v38b symbolic result is strong but cannot validate LoRA selector
- `BUILD_LIVE_NL_PARSER` — for live drop-in, create NL→predicate/FOL parser


In [ ]:

if health.get('healthy_for_lora_selector'):
    decision = {
        'decision': 'RUN_SELECTOR_ON_VALID_LORA_PREDS_OR_USE_CURRENT_SELECTED_IF_ALREADY_VALID',
        'reason': 'Generation health passes; selected metrics can be trusted as LoRA+v38b selector.'
    }
else:
    decision = {
        'decision': 'DO_NOT_TRUST_CURRENT_SELECTED_AS_LORA_SELECTOR; USE_V38B_SYMBOLIC_REPORT_AND_RERUN_GENERATION_OR_USE_OLD_VALID_PREDS',
        'reason': f"Generation unhealthy: gen_error_rate={health.get('gen_error_rate')}. v38b symbolic report remains valid, but LoRA baseline/delta is invalid."
    }

# Always add strategic next step.
decision['next_best_steps'] = [
    'For offline: run selector on a valid full LoRA preds file, or fix generation and rerun.',
    'For live: do not require server for audit; build NL->predicate/FOL parser before using v38b live.',
    'For reporting: use audit_overfit_underfit_sota_report.json and audit_sota_comparison.json.'
]

print(json.dumps(decision, indent=2, ensure_ascii=False))
write_json(decision, OUT_DIR/'audit_final_decision.json')
